# liveflights GRU trajectory model (Colab)
Trains the GRU on windows built by `build_windows_v2.py`. Runtime: **GPU** (Runtime > Change runtime type > T4).

Before this: upload to Google Drive folder `MyDrive/liveflights/`
- `gru_v2/`  (the dataset folder: train/, val/, test/, dataset.json)
- `train_gru_v2.py`  (from `ml/scratch/`)

In [ ]:
import torch, subprocess
print(torch.__version__, 'cuda:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/liveflights'
!ls -la $DRIVE

Copy the dataset to the fast local disk first (reading 3 GB from Drive during training is slow).

In [ ]:
!mkdir -p /content/gru_v2 /content/runs
!cp -r $DRIVE/gru_v2/* /content/gru_v2/
!cp $DRIVE/train_gru_v2.py /content/
!du -sh /content/gru_v2/*; cat /content/gru_v2/dataset.json | head -5

## Variant B (core + extra fields + masks): the model we want to serve

In [ ]:
!python /content/train_gru_v2.py --data /content/gru_v2 --variant B --epochs 15 --out /content/runs/B 2>&1 | tee /content/runs/B.log

## Variant A (core fields only): the ablation. B must beat A, otherwise the extra fields are not worth it

In [ ]:
!python /content/train_gru_v2.py --data /content/gru_v2 --variant A --epochs 15 --out /content/runs/A 2>&1 | tee /content/runs/A.log

## Learning curve (is more data still helping?)  train on 25% / 50% / 100% of the windows

In [ ]:
for f in (0.25, 0.5):
    !python /content/train_gru_v2.py --data /content/gru_v2 --variant B --epochs 15 --train-frac $f --out /content/runs/B_frac$f 2>&1 | tail -25

In [ ]:
import json
for name in ['A','B']:
    m = json.load(open(f'/content/runs/{name}/metrics.json'))
    print(name, 'test all      :', {k: round(v,3) for k,v in m['test']['all'].items() if k!='n'})
    print(name, 'test turning  :', {k: round(v,3) for k,v in m['test']['turning'].items() if k!='n'})

Save everything back to Drive (the Colab disk is wiped when the session ends).

In [ ]:
!mkdir -p $DRIVE/runs && cp -r /content/runs/* $DRIVE/runs/ && ls $DRIVE/runs